# 6.9 Python

Chapter 6 develops the theory of moments — expected values of powers of a random variable — and introduces the moment generating function (MGF) as a tool for computing them. This section uses Python to define functions, evaluate moments numerically via integration and summation, compute sample statistics, locate medians and modes, and work with the Log-Normal and Weibull distributions. A simulation of rolling six dice closes the section.

In [ ]:
import numpy as np
import scipy.stats as st
import scipy.integrate as integrate
import scipy.optimize as optimize
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(palette="deep")
rng = np.random.default_rng(seed=42)

## Functions

The MGF of a random variable is a function of a real variable $t$, making it a natural context for discussing how Python handles functions. As a concrete example, the $N(0, 1)$ MGF is

$$M(t) = e^{t^2/2}.$$

In Python, the `def` keyword defines a function. A function of one variable `t` — the *argument* — can be written as:

In [ ]:
def M(t):
    return np.exp(t**2 / 2)

# Evaluate at a single point
print(M(0))

# Evaluate at 1, 2, ..., 10 by passing an array
print(M(np.arange(1, 11)))

Because `np.exp` operates element-wise on arrays, `M` accepts a scalar or an array with no extra work. This makes it easy to plot: evaluate on a fine grid and connect the points.

In [ ]:
t_grid = np.linspace(-3, 3, 500)

fig, ax = plt.subplots(figsize=(6, 4))
sns.lineplot(x=t_grid, y=M(t_grid), ax=ax)
ax.set_xlabel("t")
ax.set_ylabel("M(t)")
ax.set_title(r"$N(0,1)$ MGF: $M(t) = e^{t^2/2}$")
plt.tight_layout()
plt.show()

The name given to the argument inside `def` has no effect on how the function is called; writing `def M(x)` instead would define the same function. Naming arguments becomes more important for functions of several variables, because Python then allows arguments to be passed by name rather than by position. The $N(\mu, \sigma^2)$ MGF,

$$g(t) = \exp\!\left(\mu t + \frac{\sigma^2 t^2}{2}\right),$$

depends on $t$, $\mu$, and $\sigma$. We can define it with default values for the mean and standard deviation:

In [ ]:
def g(t, mean=0, sd=1):
    return np.exp(mean * t + sd**2 * t**2 / 2)

# N(2, 9) MGF at t = 1: positional and keyword forms are equivalent
print(g(1, 2, 3))
print(g(t=1, mean=2, sd=3))
print(g(mean=2, sd=3, t=1))

# N(0, 25) MGF at t = 3: omit mean, which defaults to 0
print(g(t=3, sd=5))

All three calls for the $N(2, 9)$ case produce the same value. Using keyword arguments eliminates any ambiguity about which parameter is which — a significant advantage when working with many functions over time. The default values let us call `g(t=3, sd=5)` without specifying `mean`, because the default of `0` is exactly what we want.

## Moments

The Law of the Unconscious Statistician (LOTUS) expresses any moment of a continuous random variable as an integral. For a function $h$ of $X$,

$$E[h(X)] = \int_{-\infty}^{\infty} h(x)\, f(x)\, dx,$$

where $f$ is the PDF of $X$. `scipy.integrate.quad` evaluates such integrals numerically. As an example, the 6th moment of $X \sim N(0, 1)$ is

$$E(X^6) = \int_{-\infty}^{\infty} x^6 \varphi(x)\, dx,$$

where $\varphi$ is the standard normal PDF. The true answer, derivable from the MGF, is $15$.

In [ ]:
result, error = integrate.quad(lambda x: x**6 * st.norm.pdf(x), -np.inf, np.inf)
print(f"6th moment of N(0, 1): {result:.4f}  (absolute error < {error:.2e})")

Using `np.inf` as the integration limit is preferable to a large finite number; numerical quadrature over a truncated range can substantially undercount the tails.

As a second example, the 2nd moment (equivalently, the variance) of $X \sim \mathrm{Unif}(-1, 1)$ is $1/3$.

In [ ]:
result, error = integrate.quad(
    lambda x: x**2 * st.uniform.pdf(x, loc=-1, scale=2), -1, 1
)
print(f"2nd moment of Unif(-1, 1): {result:.6f}  (true value: {1/3:.6f})")

For a discrete random variable, LOTUS replaces the integral with a sum. The 2nd moment of $X \sim \mathrm{Pois}(7)$ satisfies

$$E(X^2) = \mathrm{Var}(X) + (EX)^2 = 7 + 49 = 56.$$

We can verify this by summing $k^2 \cdot P(X = k)$ up to a large enough $k$.

In [ ]:
k = np.arange(0, 101)
second_moment = np.sum(k**2 * st.poisson.pmf(k, mu=7))
print(f"2nd moment of Pois(7): {second_moment:.6f}  (true value: 56)")

Summing to $k = 100$ is safe here because the Poisson PMF with $\mu = 7$ becomes negligible well before that point. (For the continuous case, `quad` handles the tail automatically, which is why it is preferred over an artificial upper limit.)

## Sample Moments

The $n$th *sample moment* of a dataset is the average of the $n$th powers of the observations. For a vector `x` of data, `np.mean(x**n)` computes this in one step. To check how well sample moments approximate the true moments, we generate 100 i.i.d. $N(0, 1)$ draws and compute the 6th sample moment.

In [ ]:
x = rng.normal(size=100)
print(f"6th sample moment (n=100): {np.mean(x**6):.4f}")
print(f"True 6th moment:           {15:.4f}")

With only 100 draws the sample moment fluctuates around the true value. Larger samples converge more reliably, as the law of large numbers guarantees. The sample mean and sample variance are the most commonly used sample moments. For a large sample the sample variance should be close to the true variance of the distribution.

In [ ]:
z = rng.normal(size=1000)
print(f"Sample mean:     {np.mean(z):.4f}  (true: 0)")
print(f"Sample variance: {np.var(z, ddof=1):.4f}  (true: 1)")
print(f"Sample std dev:  {np.std(z, ddof=1):.4f}  (true: 1)")

The `ddof=1` argument divides by $n - 1$ rather than $n$, producing the *unbiased* sample variance. With only one observation the denominator would be zero, so `np.var` with `ddof=1` returns `nan` in that case — consistent with the fact that a single data point tells us nothing about the spread of the population.

Python's standard libraries do not include sample skewness or sample kurtosis, but they are straightforward to define using the same pattern of standardizing by the sample standard deviation.

In [ ]:
def skew(x):
    central = np.mean((x - np.mean(x))**3)
    return central / np.std(x, ddof=1)**3

def kurt(x):
    central = np.mean((x - np.mean(x))**4)
    return central / np.std(x, ddof=1)**4 - 3

z = rng.normal(size=10000)
print(f"Sample skewness of N(0, 1): {skew(z):.4f}  (true: 0)")
print(f"Sample kurtosis of N(0, 1): {kurt(z):.4f}  (true: 0)")

## Medians and Modes

The median of a continuous distribution with CDF $F$ is the value $m$ satisfying $F(m) = 1/2$, i.e., the root of $F(x) - 1/2$. `scipy.optimize.brentq` finds a root bracketed by a given interval. For $X \sim \mathrm{Expo}(1)$, the median is $\log 2 \approx 0.693$.

In [ ]:
median_expo = optimize.brentq(lambda x: st.expon.cdf(x) - 0.5, a=0, b=2)
print(f"Median of Expo(1) via root-finding: {median_expo:.6f}")
print(f"True value (log 2):                 {np.log(2):.6f}")

# The quantile function (inverse CDF) gives the same result directly
print(f"Median via ppf:                     {st.expon.ppf(0.5):.6f}")

For this distribution the exact answer is available, so root-finding is more instructive than necessary. In general, the quantile function `ppf` (percent-point function, or inverse CDF) is the fastest route when `scipy.stats` provides one. The root-finding approach handles cases where no closed-form quantile exists.

For modes, `scipy.optimize.minimize_scalar` locates the peak of the PDF over a specified interval. The $\mathrm{Gamma}(6, 1)$ distribution has a PDF proportional to $x^5 e^{-x}$, and calculus gives a mode at $x = 5$.

In [ ]:
# Negate the PDF to turn maximization into minimization
result = optimize.minimize_scalar(
    lambda x: -st.gamma.pdf(x, a=6, scale=1),
    bounds=(0, 20), method="bounded"
)
print(f"Mode of Gamma(6, 1): {result.x:.6f}  (true: 5)")

For a discrete distribution, the median and mode can be found by scanning the PMF and CDF arrays directly. An interesting fact about the $\mathrm{Bin}(n, p)$ distribution is that when the mean $np$ is an integer, the median and mode both equal $np$. We can verify this for $\mathrm{Bin}(50, 0.2)$, whose mean is $50 \times 0.2 = 10$.

In [ ]:
n, p = 50, 0.2
k_vals = np.arange(0, n + 1)

# Median: first k where the CDF reaches or exceeds 0.5
cdf_vals = st.binom.cdf(k_vals, n=n, p=p)
median_bin = k_vals[np.argmax(cdf_vals >= 0.5)]
print(f"Median of Bin(50, 0.2): {median_bin}  (mean np = {n*p:.0f})")

# Mode: k that maximizes the PMF
pmf_vals = st.binom.pmf(k_vals, n=n, p=p)
mode_bin = k_vals[np.argmax(pmf_vals)]
print(f"Mode of Bin(50, 0.2):   {mode_bin}  (mean np = {n*p:.0f})")

`np.argmax` returns the index of the first maximum, which is exactly what we want. In a 0-indexed array starting at $k = 0$, index 10 corresponds to $k = 10$, confirming the theoretical prediction.

The sample median of a dataset is `np.median(x)`. The sample mode requires more care: `np.unique` paired with the `return_counts` option lists every distinct value and how often it appears.

In [ ]:
def sample_mode(x):
    values, counts = np.unique(x, return_counts=True)
    max_count = counts.max()
    return values[counts == max_count]

# Draw 200 values from Bin(10, 0.3) and find the sample median and mode
data = rng.binomial(n=10, p=0.3, size=200)
print(f"Sample median: {np.median(data)}")
print(f"Sample mode:   {sample_mode(data)}")

The function returns an array to handle ties naturally: if two values share the highest count, both appear. For this draw both the median and mode should be near $np = 3$.

## Log-Normal and Weibull Distributions

A random variable $X$ follows the Log-Normal distribution $LN(\mu, \sigma^2)$ when $\log X \sim N(\mu, \sigma^2)$. Equivalently, $X = e^Y$ for $Y \sim N(\mu, \sigma^2)$. In `scipy.stats`, `st.lognorm(s=sigma, scale=np.exp(mu))` gives the $LN(\mu, \sigma^2)$ distribution. The parameter `s` is the standard deviation $\sigma$ of the underlying Normal (not the variance), and `scale` is $e^\mu$.

In [ ]:
mu, sigma = 1, 2
lnorm = st.lognorm(s=sigma, scale=np.exp(mu))   # LN(mu=1, sigma^2=4)

x = 2.0
print(f"LN(1, 4) PDF at x=2: {lnorm.pdf(x):.6f}")
print(f"LN(1, 4) CDF at x=2: {lnorm.cdf(x):.6f}")

# Generate LN(1, 4) two ways: direct and via exp(Normal)
draws_direct     = lnorm.rvs(size=5, random_state=rng)
draws_via_normal = np.exp(rng.normal(loc=mu, scale=sigma, size=5))

print("\nDirect draws:          ", np.round(draws_direct, 4))
print("Via exp(Normal) draws: ", np.round(draws_via_normal, 4))

The two sets of draws differ because they use independent calls to the random generator, but both sequences are realizations from $LN(1, 4)$. The exponential-of-Normal construction makes the relationship between the two distributions concrete.

For the Weibull distribution $\mathrm{Wei}(\lambda, \gamma)$ with PDF

$$f(x) = \gamma\lambda\, x^{\gamma-1}\, e^{-\lambda x^\gamma}, \quad x > 0,$$

`scipy.stats` uses a shape–scale parameterization via `st.weibull_min(c=gamma, scale=lam**(-1/gamma))`. The Weibull is closely related to the Exponential: if $X \sim \mathrm{Expo}(\lambda)$, then $X^{1/\gamma} \sim \mathrm{Wei}(\lambda, \gamma)$.

In [ ]:
lam, gamma = 1, 2
weib = st.weibull_min(c=gamma, scale=lam**(-1 / gamma))   # Wei(lam=1, gamma=2)

x = 1.0
print(f"Wei(1, 2) PDF at x=1: {weib.pdf(x):.6f}")
print(f"Wei(1, 2) CDF at x=1: {weib.cdf(x):.6f}")

# Generate Wei(lam, gamma) two ways: direct and via Expo^(1/gamma)
draws_direct   = weib.rvs(size=5, random_state=rng)
expo_draws     = rng.exponential(scale=1 / lam, size=5)
draws_via_expo = expo_draws ** (1 / gamma)

print("\nDirect draws:             ", np.round(draws_direct, 4))
print("Via Expo^(1/gamma) draws: ", np.round(draws_via_expo, 4))

## Dice Simulation

In Section 6.7 it was shown, through a careful combinatorial argument, that the probability of rolling a total of 18 with six fair dice is $3431/6^6 \approx 0.07354$. Simulation offers a much quicker path to the same approximate answer. We generate one million independent rolls of six dice and record what fraction yield a sum of 18.

In [ ]:
n_sims = 10**6
# Each row is one roll of six dice; rng.integers(1, 7) gives 1, 2, ..., 6
dice = rng.integers(1, 7, size=(n_sims, 6))
totals = dice.sum(axis=1)

prob_18 = np.mean(totals == 18)
exact   = 3431 / 6**6
print(f"Simulated P(sum = 18): {prob_18:.5f}")
print(f"Exact      P(sum = 18): {exact:.5f}")